# M4.5 · Embedding ve BPE tokenization

Metni sayılara, sayıları vektörlere çevirmek. Byte-pair encoding'in ilk birleştirmelerini elle buluyor, saf Python'da BPE eğitici ve kodlayıcı yazıyor, NumPy'da embedding'in ileri/geri geçişini (scatter-add) ve çift sayımını döngüsüz yapıyor, PyTorch'ta `nn.Embedding` ile bir bigram dil modeli eğitip sayım tabanlı modelle aynı kayba indiğini gösteriyor ve en sonda `embedding_bag`'i segment toplamıyla yeniden kuruyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-5-embed-bpe.html](https://m-gonul.github.io/pytorch-lab/m4-5-embed-bpe.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_5_embed_bpe.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    wf = {\"low\": 5, \"lower\": 2, \"newest\": 6, \"widest\": 3}\n\n    def pc():\n        c = need(ns, \"pair_counts\")({tuple(w): f for w, f in wf.items()})\n        ensure(c[(\"e\", \"s\")] == 9 and c[(\"w\", \"e\")] == 8 and c[(\"l\", \"o\")] == 7, \"çift sayıları (e,s)=9, (w,e)=8, (l,o)=7\")\n    R.test(\"pair_counts\", pc)\n\n    def mv():\n        v = need(ns, \"merge_vocab\")({(\"a\", \"a\", \"a\"): 1, (\"a\", \"b\", \"a\", \"a\"): 2}, (\"a\", \"a\"))\n        ensure(v == {(\"aa\", \"a\"): 1, (\"a\", \"b\", \"aa\"): 2}, f\"örtüşmeyen soldan-sağa birleştirme: {v}\")\n    R.test(\"merge_vocab (aaa → aa, a)\", mv)\n\n    def tr():\n        m = need(ns, \"train_bpe\")(wf, 4)\n        ensure([tuple(p) for p in m] == [(\"e\", \"s\"), (\"es\", \"t\"), (\"l\", \"o\"), (\"lo\", \"w\")], f\"birleştirmeler {m}\")\n        ensure(len(need(ns, \"train_bpe\")({\"ab\": 1}, 5)) == 1, \"çift kalmayınca durmalı\")\n    R.test(\"train_bpe (eşitlik kuralı dahil)\", tr)\n\n    def enc():\n        m = need(ns, \"train_bpe\")(wf, 4)\n        ensure(list(need(ns, \"encode\")(\"lowest\", m)) == [\"low\", \"est\"], \"lowest → low, est\")\n        ensure(list(need(ns, \"encode\")(\"newer\", m)) == [\"n\", \"e\", \"w\", \"e\", \"r\"], \"newer birleşmez\")\n    R.test(\"encode\", enc)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef py_merge(ids, a, b, new):\n    out, i = [], 0\n    while i < len(ids):\n        if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:\n            out.append(new); i += 2\n        else:\n            out.append(ids[i]); i += 1\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n\n    def eb():\n        close(need(ns, \"embed_backward\")(np.array([2, 0, 2]), np.array([[1.0, 1], [2, 2], [3, 3]]), 4), [[2, 2], [0, 0], [4, 4], [0, 0]])\n        ids = rng.integers(0, 10, 200); dout = rng.normal(size=(200, 3))\n        ref = np.zeros((10, 3))\n        for i, d in zip(ids, dout):\n            ref[i] += d\n        close(need(ns, \"embed_backward\")(ids, dout, 10), ref, what=\"200 id\")\n    R.test(\"embed_backward: tekrarlı id'ler toplanıyor\", eb)\n\n    def pcn():\n        ids = rng.integers(0, 6, 300)\n        C = need(ns, \"pair_counts_np\")(ids, 6)\n        ref = np.zeros((6, 6), int)\n        for a, b in zip(ids[:-1], ids[1:]):\n            ref[a, b] += 1\n        ensure(np.array_equal(C, ref), \"çift sayım matrisi farklı\")\n    R.test(\"pair_counts_np\", pcn)\n\n    def mg():\n        ensure(list(need(ns, \"merge_ids\")(np.array([0, 1, 2, 0, 1, 1, 0, 1]), 0, 1, 9)) == [9, 2, 9, 1, 9], \"elle örnek\")\n        for _ in range(5):\n            ids = rng.integers(0, 4, 100)\n            ensure(list(need(ns, \"merge_ids\")(ids, 1, 2, 7)) == py_merge(list(ids), 1, 2, 7), \"rastgele dizi\")\n    R.test(\"merge_ids (a ≠ b)\", mg)\n",
 "d": "import math\nimport torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    text = (\"ali ata bak. ayse ip atla. ali topu at. ayse kitap oku. ali ile ayse parka gitti. \"\n            \"kedi agaca cikti. kopek havladi. kus ucup gitti. \") * 12\n    chars = sorted(set(text)); V = len(chars)\n    ids = torch.tensor([chars.index(c) for c in text])\n\n    def cn():\n        N = torch.zeros(V, V)\n        for a, b in zip(ids[:-1].tolist(), ids[1:].tolist()):\n            N[a, b] += 1\n        ref = -sum(math.log(N[a, b] / N[a].sum()) for a, b in zip(ids[:-1].tolist(), ids[1:].tolist())) / (len(ids) - 1)\n        close(need(ns, \"count_nll\")(ids, V), ref, rtol=1e-5)\n    R.test(\"count_nll = sayım modelinin NLL'i\", cn)\n\n    def tr():\n        torch.manual_seed(0)\n        m, loss = need(ns, \"train_bigram\")(ids, V, steps=500, lr=0.5)\n        ensure(isinstance(m, nn.Module) and any(isinstance(x, nn.Embedding) for x in m.modules()), \"nn.Embedding içeren bir model bekleniyordu\")\n        opt = need(ns, \"count_nll\")(ids, V)\n        ensure(abs(loss - opt) < 0.02, f\"sinir ağı kaybı {loss:.4f}, sayım optimumu {opt:.4f}\")\n        return f\"sinir ağı {loss:.4f} ≈ sayım MLE {opt:.4f} (aynı model, iki yol)\"\n    R.test(\"bigram ağı sayım MLE'sine yakınsıyor\", tr)\n\n    def sparse():\n        torch.manual_seed(0)\n        m = need(ns, \"BigramLM\", \"sınıf\")(V)\n        x = torch.tensor([3, 3, 5]); y = torch.tensor([1, 2, 0])\n        torch.nn.functional.cross_entropy(m(x), y).backward()\n        W = [p for p in m.parameters()][0]\n        rows = (W.grad.abs().sum(1) > 0).nonzero().view(-1).tolist()\n        ensure(rows == [3, 5], f\"yalnız kullanılan satırlar gradyan almalı: {rows}\")\n    R.test(\"embedding gradyanı yalnız kullanılan satırlarda\", sparse)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    W = torch.randn(50, 8, generator=g, dtype=torch.float64)\n\n    def eq():\n        ids = torch.randint(0, 50, (40,), generator=g)\n        offsets = torch.tensor([0, 3, 3, 10, 25, 39])\n        close(need(ns, \"embedding_bag_mean\")(W, ids, offsets), F.embedding_bag(ids, W, offsets, mode=\"mean\"))\n    R.test(\"F.embedding_bag(mode='mean') ile aynı (boş grup dahil)\", eq)\n\n    def gr():\n        Wa = W.clone().requires_grad_(True); Wb = W.clone().requires_grad_(True)\n        ids = torch.randint(0, 50, (30,), generator=g); off = torch.tensor([0, 7, 20])\n        need(ns, \"embedding_bag_mean\")(Wa, ids, off).pow(2).sum().backward()\n        F.embedding_bag(ids, Wb, off, mode=\"mean\").pow(2).sum().backward()\n        close(Wa.grad, Wb.grad)\n    R.test(\"gradyan da aynı\", gr)\n    if \"embedding_bag_mean\" not in ns:\n        return\n    Wb = torch.randn(10000, 64, generator=g)\n    lens = torch.randint(1, 50, (2000,), generator=g)\n    off = torch.cat([torch.zeros(1, dtype=torch.long), lens.cumsum(0)[:-1]])\n    ids = torch.randint(0, 10000, (int(lens.sum()),), generator=g)\n    t_loop = bench(lambda: torch.stack([Wb[ids[o:o + l]].mean(0) for o, l in zip(off.tolist(), lens.tolist())]), repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"embedding_bag_mean\"](Wb, ids, off), repeat=3)\n    t_ref = bench(lambda: F.embedding_bag(ids, Wb, off, mode=\"mean\"), repeat=3)\n    R.table(\"2000 torba, ort. 25 token, d = 64\", [\"yöntem\", \"süre\"],\n            [[\"for torba: mean\", fmt_time(t_loop)], [\"senin embedding_bag_mean\", fmt_time(t_vec)], [\"F.embedding_bag (fused)\", fmt_time(t_ref)]],\n            note=\"Fused kernel ara (toplam id × d) tensörünü (weight[ids]) hiç oluşturmaz.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m4_5_embed_bpe as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir dil modeli metni önce **token** dizisine çevirir. Byte-pair encoding:

1. Her kelimeyi karakterlerine böl (GPT-2'de baytlarına).
2. Tüm komşu sembol çiftlerini frekansla say.
3. En sık çifti yeni bir sembole birleştir; birleştirme listesine ekle.
4. İstenen sözlük boyutuna kadar 2–3'ü tekrarla.

Kodlama: yeni bir kelimeyi karakterlere böl ve öğrenilen birleştirmeleri **aynı sırayla** uygula.

Her token id'si bir **embedding** vektörüne eşlenir: $E\in\mathbb R^{V\times d}$, $e=E[\text{id}]$. Bu, one-hot vektörle matris çarpımıyla aynıdır, $e=\text{onehot}(\text{id})^\top E$, ama $V$ kat daha ucuzdur. Gradyan ise kullanılan satırlara **toplanarak** dağılır:

$$
\frac{\partial L}{\partial E_v}=\sum_{t:\ \text{id}_t=v}\frac{\partial L}{\partial e_t}\qquad(\text{scatter-add}).
$$

## 1 · Elle

<!-- çözüm -->

Çiftleri say: $(e,s)$ ve $(s,t)$ 9'da berabere; sözlük sırası kuralıyla $(e,s)$ kazanır. Sonra "es" bir sembol olur ve $(es,t)$ 9 ile tek başına en sık çift. "newest" artık n, e, w, est.

Embedding kısmında id 2 iki kez kullanılıyor; gradyanı iki katkının toplamı. Bu, M0.3'teki "aynı düğüm iki yoldan → topla" kuralının bir başka hâli.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Kelime frekansları: low ×5, lower ×2, newest ×6, widest ×3 (karakterlere bölünmüş). Çift sayımı frekansla ağırlıklı**

- `es`: (e, s) çiftinin sayısı
- `we`: (w, e) çiftinin sayısı
- `lo`: (l, o) çiftinin sayısı

**Eşitlikte sözlük sırasında küçük çift kazanır: 1. birleştirme (e, s) → es, 2. (es, t) → est**

- `ntok`: 2 birleştirmeden sonra "newest" kaç token?
- `lowm`: "low" kaçıncı birleştirmeden sonra tek token olur?

**Embedding $E=\begin{bmatrix}0.1&0.2\\0.3&0.4\\0.5&0.6\\0.7&0.8\end{bmatrix}$**

- `look`: id = 2 için vektörün 1. elemanı (one-hot × E)
- `grad`: ids = [2, 0, 2], gelen gradyan satırları [1,1], [2,2], [3,3]: $\partial L/\partial E_{2,1}$

In [ ]:
cevaplar = dict(
    es=None,
    we=None,
    lo=None,
    ntok=None,
    lowm=None,
    look=None,
    grad=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"es":[9,0],"we":[8,0],"lo":[7,0],"ntok":[4,0],"lowm":[4,0],"look":[0.5,0.0001],"grad":[4,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. Kelimeler sembol tuple'ı olarak: `("l", "o", "w")`.

- `pair_counts(vocab)` — `vocab: {tuple: frekans}` → `{(a, b): sayı}`.
- `merge_vocab(vocab, pair)` → yeni `vocab`; her kelimede soldan sağa, örtüşmeyen birleştirme.
- `train_bpe(word_freqs, num_merges)` — `word_freqs: {"low": 5, …}` → birleştirme listesi `[("e", "s"), ("es", "t"), …]`. Eşitlikte sözlük sırasında küçük çift (`min(çiftler, key=lambda p: (-sayı, p))`). Birleştirilecek çift kalmazsa dur.
- `encode(word, merges)` → token listesi.

In [ ]:
%%writefile stage_b.py
from collections import Counter


def pair_counts(vocab):
    ...


def merge_vocab(vocab, pair):
    ...


def train_bpe(word_freqs, num_merges):
    ...


def encode(word, merges):
    ...


merges = train_bpe({"low": 5, "lower": 2, "newest": 6, "widest": 3}, 4)
print(merges)                     # [('e','s'), ('es','t'), ('l','o'), ('lo','w')]
print(encode("lowest", merges))   # ['low', 'est']

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**:

- `embed_backward(ids, dout, V)` → `dE: (V, d)`; `np.add.at(dE, ids, dout)` (aynı id'lerin katkılarını toplar; `dE[ids] += dout` toplamaz).
- `pair_counts_np(ids, V)` → `(V, V)` sayım matrisi; çiftleri tek tamsayıya kodla: `ids[:-1] * V + ids[1:]`, sonra `np.bincount(…, minlength=V*V)`.
- `merge_ids(ids, a, b, new)` — `a ≠ b` için `(a, b)` geçişlerini `new` ile değiştir (bu durumda eşleşmeler örtüşemez). Eşleşme maskesi `m = (ids[:-1] == a) & (ids[1:] == b)`; eşleşmenin ilk elemanını `new` yap, ikincisini sil.

In [ ]:
%%writefile stage_c.py
import numpy as np


def embed_backward(ids, dout, V):
    ...


def pair_counts_np(ids, V):
    ...


def merge_ids(ids, a, b, new):
    ...


print(embed_backward(np.array([2, 0, 2]), np.array([[1.0, 1], [2, 2], [3, 3]]), 4))
print(merge_ids(np.array([0, 1, 2, 0, 1, 1, 0, 1]), 0, 1, 9))   # [9, 2, 9, 1, 9]

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Karakter düzeyinde bigram dil modeli: önceki karakter $a$'ya bakarak sonraki $b$'nin dağılımı. Sayım modeli $\hat p(b\mid a)=\frac{N_{ab}}{N_a}$; sinir ağı modeli `nn.Embedding(V, V)`: her $a$ için bir logit satırı.

- `BigramLM(V)` — `nn.Module`; `forward(idx)` → `(…, V)` logit.
- `train_bigram(ids, V, steps=500, lr=0.5)` → `(model, son_kayıp)`; girdiler `ids[:-1]`, hedefler `ids[1:]`, tam batch, `Adam`.
- `count_nll(ids, V)` → sayım modelinin ortalama NLL'i (gözlenen çiftler için $-\log\frac{N_{ab}}{N_a}$ ortalaması).

Kontrol, iki kaybın birbirine yakınsadığını doğrular.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F
from torch import nn


class BigramLM(nn.Module):
    def __init__(self, V):
        super().__init__()
        ...

    def forward(self, idx):
        ...


def train_bigram(ids, V, steps=500, lr=0.5):
    ...


def count_nll(ids, V):
    ...


text = "kedi kediye bakti. kopek kediye havladi. kedi kacti. " * 20
chars = sorted(set(text))
ids = torch.tensor([chars.index(c) for c in text])
model, loss = train_bigram(ids, len(chars))
print(loss, count_nll(ids, len(chars)))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`F.embedding_bag(ids, weight, offsets, mode="mean")`: düz bir id dizisi ve her grubun başlangıç `offsets`'i; her grubun embedding ortalaması. Metin sınıflandırmada ve öneri sistemlerinde "kelime torbası" bu.

- `embedding_bag_mean(weight, ids, offsets)` → `(B, d)`. **Döngü yok:** grup uzunlukları `torch.diff(offsets, append=…)`, her id'nin grup numarası `torch.repeat_interleave(torch.arange(B), lengths)`, toplam `zeros(B, d).index_add_(0, seg, weight[ids])`. Boş grup → sıfır vektör.

In [ ]:
%%writefile stage_e.py
import torch


def embedding_bag_mean(weight, ids, offsets):
    """→ (B, d). Döngü yok."""
    ...


W = torch.arange(12.0).view(6, 2)
print(embedding_bag_mean(W, torch.tensor([0, 2, 4, 5, 1]), torch.tensor([0, 2, 2])))   # [[2,3],[0,0],[6.67,7.67]]

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Kudo & Richardson (2018): SentencePiece ve unigram LM tokenizasyonu

[Kudo & Richardson — SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing (EMNLP 2018); Kudo — Subword Regularization (ACL 2018)](https://arxiv.org/abs/1808.06226)

BPE'nin (M4.5) yanına unigram dil modeli tokenizasyonunu yaz (Kudo 2018): büyük bir aday sözlükle başla, EM ile alt kelime olasılıklarını öğren, Viterbi ile en olası bölütlemeyi bul ve kaybı en az artıran adayları budayarak sözlüğü küçült. Türkçe bir metin üzerinde BPE ile karşılaştır: kelime başına token sayısı, morfem sınırlarıyla uyum; alt kelime düzenlileştirmesini (birden çok bölütlemeden örnekleme) göster.


**Adımlar**

1. Boşlukları ▁ ile kodlayıp dil bağımsız ön işlem (SentencePiece).

**Kabul ölçütleri**

- Unigram bölütlemeleri morfem sınırlarına BPE'den daha yakın.
- Örneklenen bölütleme çeşitliliği gösterimi.

## Kaynaklar

- [Sennrich, Haddow, Birch — Neural Machine Translation of Rare Words with Subword Units (BPE)](https://arxiv.org/abs/1508.07909) — arXiv
- [Karpathy — minbpe ve Zero to Hero: makemore (bigram), Let's build the GPT Tokenizer](https://github.com/karpathy/minbpe) — MIT
- [Stanford CS336 Assignment 1 — BPE tokenizer](https://github.com/stanford-cs336/assignment1-basics) — MIT
- [Stanford CS224n — Word vectors](https://web.stanford.edu/class/cs224n/) — doğrulanmadı